<a name="top"></a><img src="../images/chisel_1024.png" alt="Chisel logo" style="width:480px;">

# 警告：此文件不是 Bootcamp 教学步骤的一部分。
# 它是一个中高级示例，展示了你将学到的内容。
# 如果你正在通过 Bootcamp 学习 Chisel，请不要从这里开始。
# 请从 [Scala 简介](1_intro_to_scala.ipynb) 开始

# Chisel 演示
**下一篇：[Scala 简介](1_intro_to_scala.ipynb)**

欢迎！也许你是一位听到"Chisel"这个名字感到好奇的学生，也许你是一位经验丰富的硬件设计老手，被经理指派去探索 Chisel 作为一种新的 HDL 替代方案。无论如何，如果你是 Chisel 的新手，你想尽快弄清楚它的魅力所在。不用再找了——让我们看看 Chisel 到底能做什么！

## 环境设置
在开始之前，我们需要下载并导入演示所需的依赖项。

**请通过按键盘上的 SHIFT+ENTER 或菜单中的 Run 按钮来运行以下两个代码块。**

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.iotesters.{ChiselFlatSpec, Driver, PeekPokeTester}

## 硬件生成器：RTL 的类型安全元编程

所有硬件描述语言都支持编写 RTL 设计的单个实例——Chisel 也不例外。
事实上，大多数 Verilog/VHDL 数字逻辑设计可以直接转录成 Chisel！
虽然 Chisel 还提供了其他很棒的特性（我们稍后会讲到），但我们想强调的是，切换到 Chisel 的用户将保留与任何其他硬件语言完全相同的设计控制能力。

以下是一个以 FIR 滤波器风格实现的 3 点移动平均示例。

<img src="images/demo_fir_filter.svg" width="512" />

Chisel 提供了与可综合 Verilog 类似的基本原语，并且*可以*这样使用！运行下一个代码块来声明我们的 Chisel 模块。

In [ ]:
// 3-point moving average implemented in the style of a FIR filter
class MovingAverage3(bitWidth: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(bitWidth.W))
    val out = Output(UInt(bitWidth.W))
  })

  val z1 = RegNext(io.in) // Create a register whose input is connected to the argument io.in
  val z2 = RegNext(z1)    // Create a register whose input is connected to the argument z1

  io.out := (io.in * 1.U) + (z1 * 1.U) + (z2 * 1.U) // `1.U` is an unsigned literal with value 1
}

定义 `class MovingAverage3` 之后，让我们实例化它并查看其结构：

In [ ]:
// same 3-point moving average filter as before
visualize(() => new MovingAverage3(8))

在这个 Chisel 实例的可视化中，左侧是输入，金色的是 z1 和 z2 寄存器。两个寄存器和 io_in 都乘以各自的系数，然后依次相加。`tail` 和 `bits` 元素用于防止加法结果不断增长。

你可能会问："好吧——你可以在 Chisel 中做 Verilog 能做的事，那我为什么要用 Chisel 呢？"

很高兴你这么问！Chisel 真正的威力在于创建**生成器，而不是实例**。假设我们不仅仅想要一个 `MovingAverage3` 模块，而是想创建一个由系数列表参数化的通用 `FIRFilter` 模块。

下面我们将 `MovingAverage3` 改写为接受一个系数序列。系数的数量将决定滤波器的大小。

In [ ]:
// Generalized FIR filter parameterized by the convolution coefficients
class FirFilter(bitWidth: Int, coeffs: Seq[UInt]) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(bitWidth.W))
    val out = Output(UInt())
  })
  // Create the serial-in, parallel-out shift register
  val zs = Reg(Vec(coeffs.length, UInt(bitWidth.W)))
  zs(0) := io.in
  for (i <- 1 until coeffs.length) {
    zs(i) := zs(i-1)
  }

  // Do the multiplies
  val products = VecInit.tabulate(coeffs.length)(i => zs(i) * coeffs(i))

  // Sum up the products
  io.out := products.reduce(_ +& _)
}

现在，通过在实例化时更改 `coeffs` 参数，我们的 `FIRFilter` 模块可以用来实例化无数不同的硬件模块！下面我们创建了三个不同的 `FIRFilter` 实例：

In [ ]:
// same 3-point moving average filter as before
visualize(() => new FirFilter(8, Seq(1.U, 1.U, 1.U)))

In [ ]:
// 1-cycle delay as a FIR filter
visualize(() => new FirFilter(8, Seq(0.U, 1.U)))

In [ ]:
// 5-point FIR filter with a triangle impulse response
visualize(() => new FirFilter(8, Seq(1.U, 2.U, 3.U, 2.U, 1.U)))

没有这种强大的参数化能力，我们就需要更多的模块定义——可能每种 FIR 滤波器都需要一个。理想情况下，我们希望生成器是 (1) 可组合的，(2) 强大的，(3) 能够对生成的设计进行细粒度控制。

Chisel 的优势在于你如何使用它，而不在于语言本身。
如果你决定编写实例而不是生成器，你会发现 Chisel 相对于 Verilog 的优势并不明显。
但如果你花时间学习如何编写生成器，那么 Chisel 的威力就会显现出来，你会发现再也回不去写 Verilog 的日子了。
学习编写生成器是困难的，但我们希望本教程能为你铺平道路，让你成为更好的硬件设计师、程序员和思想者！

---
# 全部完成！

[返回顶部。](#top)